_This Notebook resamples the the Sentinel-1 data rasters to quartal aggregations._

## Importing functions

In [1]:
from notebooks_dir._01_pre_processing_sat_obs._support._n04_funcs import (
    resample_stitched_images_tree,
    calculate_vv_vh_ratio_tree,
    calculate_quarterly_std_tree,
    calculate_monthly_std_tree,
    calculate_quarterly_means_tree,
    convert_quarterly_to_db,
    convert_monthly_to_db,
)

## Setting directory

In [2]:
from paths.OG_paths import (
    s1_mosaics__n2000__clip__dir,
    s1_mosaic_10m_res__dir,
    s1_Q_mosaic__dir,
    s1_Q_mosaic_dB__dir,
)

# Sentinel-1 Quarterly Feature Extraction Explanation

# Sentinel-1 Quarterly Feature Extraction

## Overview
From 3 monthly Sentinel-1 mosaics per quarter, 5 features that capture backscatter magnitude, scattering mechanisms, and temporal variability are derived.

## Feature Extraction Workflow

### 1. **VV_mean (dB)**
- **Process**: Average in linear domain, then convert to dB
- **Steps**:
  1. Take 3 monthly VV mosaics (linear power values)
  2. Calculate mean: `VV_mean_lin = mean(VV₁, VV₂, VV₃)`
  3. Convert to dB: `VV_mean_dB = 10 × log₁₀(VV_mean_lin)`
- **Interpretation**: Typical VV backscatter level for the quarter

### 2. **VH_mean (dB)**
- **Process**: Average in linear domain, then convert to dB
- **Steps**:
  1. Take 3 monthly VH mosaics (linear power values)
  2. Calculate mean: `VH_mean_lin = mean(VH₁, VH₂, VH₃)`
  3. Convert to dB: `VH_mean_dB = 10 × log₁₀(VH_mean_lin)`
- **Interpretation**: Typical VH backscatter level for the quarter

### 3. **VV/VH_ratio (dB)**

**Option 1: Ratio of means (chosen)**

VV_mean_lin = mean(VV₁, VV₂, VV₃)
VH_mean_lin = mean(VH₁, VH₂, VH₃)
ratio_dB = 10 × log₁₀(VV_mean_lin / VH_mean_lin)

**Option 2: Mean of ratios**

ratio₁_lin = VV₁ / VH₁
ratio₂_lin = VV₂ / VH₂
ratio₃_lin = VV₃ / VH₃
ratio_mean_lin = mean(ratio₁, ratio₂, ratio₃)
ratio_dB = 10 × log₁₀(ratio_mean_lin)

- **Interpretation**: Dominant scattering mechanism (surface vs. volume scattering)

### 4. **VV_std (dB)**
- **Process**: Standard deviation calculated in dB domain
- **Steps**:
  1. Convert monthly mosaics to dB: `VV₁_dB, VV₂_dB, VV₃_dB`
  2. Calculate std: `VV_std = std(VV₁_dB, VV₂_dB, VV₃_dB)`
- **Interpretation**: Temporal variability of VV backscatter (relative changes)

### 5. **VH_std (dB)**
- **Process**: Standard deviation calculated in dB domain
- **Steps**:
  1. Convert monthly mosaics to dB: `VH₁_dB, VH₂_dB, VH₃_dB`
  2. Calculate std: `VH_std = std(VH₁_dB, VH₂_dB, VH₃_dB)`
- **Interpretation**: Temporal variability of VH backscatter (relative changes)

## Final Feature Set

| Feature | Unit | Captures |
|---------|------|----------|
| VV_mean | dB | Baseline backscatter (moisture/roughness) |
| VH_mean | dB | Baseline backscatter (volume scattering) |
| VV/VH_ratio | dB | Scattering mechanism |
| VV_std | dB | Temporal dynamics (wetting/drying cycles) |
| VH_std | dB | Temporal dynamics (vegetation changes) |

## Key Principles
- **Means**: Calculated in linear domain (physically correct averaging of power)
- **Std**: Calculated in dB domain (captures relative variability)
- **Ratio**: Derived from means (consistent with averaging approach)
- **Output**: All features in dB for consistent scaling in Random Forest model

# Workflow

## 0) Resampling to 10m resolution

In [3]:
processed, skipped = resample_stitched_images_tree(
    in_root=s1_mosaics__n2000__clip__dir,
    out_root=s1_mosaic_10m_res__dir,
    target_resolution=10.0,
    resampling_method="bilinear",
    band_pattern="VV.tif",
)


Found 107 files matching 'VV.tif'
  - VV.tif: resampled 20.0m -> 10.0m (2351x3436 -> 4702x6872) -> VV_10m.tif
  - VV.tif: resampled 20.0m -> 10.0m (2351x3436 -> 4702x6872) -> VV_10m.tif
  - VV.tif: resampled 20.0m -> 10.0m (2351x3436 -> 4702x6872) -> VV_10m.tif
  - VV.tif: resampled 20.0m -> 10.0m (2351x3436 -> 4702x6872) -> VV_10m.tif
  - VV.tif: resampled 20.0m -> 10.0m (2351x3436 -> 4702x6872) -> VV_10m.tif
  - VV.tif: resampled 20.0m -> 10.0m (2351x3436 -> 4702x6872) -> VV_10m.tif
  - VV.tif: resampled 20.0m -> 10.0m (2351x3436 -> 4702x6872) -> VV_10m.tif
  - VV.tif: resampled 20.0m -> 10.0m (2351x3436 -> 4702x6872) -> VV_10m.tif
  - VV.tif: resampled 20.0m -> 10.0m (2351x3436 -> 4702x6872) -> VV_10m.tif
  - VV.tif: resampled 20.0m -> 10.0m (2351x3436 -> 4702x6872) -> VV_10m.tif
  - VV.tif: resampled 20.0m -> 10.0m (2351x3436 -> 4702x6872) -> VV_10m.tif
  - VV.tif: resampled 20.0m -> 10.0m (2351x3436 -> 4702x6872) -> VV_10m.tif
  - VV.tif: resampled 20.0m -> 10.0m (2351x3436 -> 47

In [4]:
processed, skipped = resample_stitched_images_tree(
    in_root=s1_mosaics__n2000__clip__dir,
    out_root=s1_mosaic_10m_res__dir,
    target_resolution=10.0,
    resampling_method="bilinear",
    band_pattern="VH.tif",
)


Found 107 files matching 'VH.tif'
  - VH.tif: resampled 20.0m -> 10.0m (2351x3436 -> 4702x6872) -> VH_10m.tif
  - VH.tif: resampled 20.0m -> 10.0m (2351x3436 -> 4702x6872) -> VH_10m.tif
  - VH.tif: resampled 20.0m -> 10.0m (2351x3436 -> 4702x6872) -> VH_10m.tif
  - VH.tif: resampled 20.0m -> 10.0m (2351x3436 -> 4702x6872) -> VH_10m.tif
  - VH.tif: resampled 20.0m -> 10.0m (2351x3436 -> 4702x6872) -> VH_10m.tif
  - VH.tif: resampled 20.0m -> 10.0m (2351x3436 -> 4702x6872) -> VH_10m.tif
  - VH.tif: resampled 20.0m -> 10.0m (2351x3436 -> 4702x6872) -> VH_10m.tif
  - VH.tif: resampled 20.0m -> 10.0m (2351x3436 -> 4702x6872) -> VH_10m.tif
  - VH.tif: resampled 20.0m -> 10.0m (2351x3436 -> 4702x6872) -> VH_10m.tif
  - VH.tif: resampled 20.0m -> 10.0m (2351x3436 -> 4702x6872) -> VH_10m.tif
  - VH.tif: resampled 20.0m -> 10.0m (2351x3436 -> 4702x6872) -> VH_10m.tif
  - VH.tif: resampled 20.0m -> 10.0m (2351x3436 -> 4702x6872) -> VH_10m.tif
  - VH.tif: resampled 20.0m -> 10.0m (2351x3436 -> 47

In [5]:
processed, skipped = calculate_vv_vh_ratio_tree(
    in_root=s1_mosaic_10m_res__dir,
    out_root=s1_mosaic_10m_res__dir,
    vv_pattern="VV*.tif",
    vh_pattern="VH*.tif",
)


SUCCESS 2017_01: VV_10m.tif / VH_10m.tif -> VV_VH_ratio.tif (min=0.13, max=3872.67, mean=5.21)

SUCCESS 2017_02: VV_10m.tif / VH_10m.tif -> VV_VH_ratio.tif (min=0.09, max=2570.89, mean=5.23)

SUCCESS 2017_03: VV_10m.tif / VH_10m.tif -> VV_VH_ratio.tif (min=0.09, max=2502.75, mean=5.03)

SUCCESS 2017_04: VV_10m.tif / VH_10m.tif -> VV_VH_ratio.tif (min=0.08, max=3671.16, mean=4.68)

SUCCESS 2017_05: VV_10m.tif / VH_10m.tif -> VV_VH_ratio.tif (min=0.10, max=3095.53, mean=4.60)

SUCCESS 2017_06: VV_10m.tif / VH_10m.tif -> VV_VH_ratio.tif (min=0.10, max=3373.50, mean=4.52)

SUCCESS 2017_07: VV_10m.tif / VH_10m.tif -> VV_VH_ratio.tif (min=0.10, max=4034.42, mean=4.42)

SUCCESS 2017_08: VV_10m.tif / VH_10m.tif -> VV_VH_ratio.tif (min=0.09, max=3318.06, mean=4.43)

SUCCESS 2017_09: VV_10m.tif / VH_10m.tif -> VV_VH_ratio.tif (min=0.08, max=2983.30, mean=4.64)

SUCCESS 2017_10: VV_10m.tif / VH_10m.tif -> VV_VH_ratio.tif (min=0.09, max=3314.34, mean=4.79)

SUCCESS 2017_11: VV_10m.tif / VH_10m.ti

"The processed data is then combined using a multitemporal compositing algorithm, which employs local resolution weighting to reduce noise and enhance spatial homogeneity. This means a weighted average is calculated from all the valid individual image pixels within the month, applying higher weighting to pixels having higher local resolution (slopes facing away from the sensor)." -> [https://dataspace.copernicus.eu/news/2024-8-6-announcing-sentinel-1-monthly-mosaics-copernicus-data-space-ecosystem] <br>

Nearly all radar signals have a co-polarized and a cross-polarized component, but the respective power of these components varies according to the nature of scattering (reflection). The radar signal can be reflected in several different ways, and each of these can be interpreted from the results. 

Using QGIS it is found that there is no true difference between the mean and medain images. It can thus be inferred that there are no true extremes in the data and the mean is the preferred option

## 1) VV_mean per Q

In [6]:
vv_quarterly_mean = calculate_quarterly_means_tree(
    in_root=s1_mosaic_10m_res__dir,
    out_root=s1_Q_mosaic__dir,
    band_pattern="VV*.tif",
    output_prefix="VV_Q",
    date_format="%Y_%m",
    aggregation="mean",
)


Found 36 quarters with data:
  2017 Q1: 3 images
  2017 Q2: 3 images
  2017 Q3: 3 images
  2017 Q4: 3 images
  2018 Q1: 3 images
  2018 Q2: 3 images
  2018 Q3: 3 images
  2018 Q4: 3 images
  2019 Q1: 3 images
  2019 Q2: 3 images
  2019 Q3: 3 images
  2019 Q4: 3 images
  2020 Q1: 3 images
  2020 Q2: 3 images
  2020 Q3: 3 images
  2020 Q4: 3 images
  2021 Q1: 3 images
  2021 Q2: 3 images
  2021 Q3: 3 images
  2021 Q4: 3 images
  2022 Q1: 3 images
  2022 Q2: 3 images
  2022 Q3: 3 images
  2022 Q4: 3 images
  2023 Q1: 3 images
  2023 Q2: 3 images
  2023 Q3: 3 images
  2023 Q4: 3 images
  2024 Q1: 3 images
  2024 Q2: 3 images
  2024 Q3: 3 images
  2024 Q4: 3 images
  2025 Q1: 3 images
  2025 Q2: 3 images
  2025 Q3: 3 images
  2025 Q4: 2 images

Processing 2017 Q1 (3 images)...


C:\Users\NL1G3K\Desktop\Vegetation_quality_monitoring\notebooks_dir\_01_pre_processing_sat_obs\_support\_n04_funcs.py:873: RuntimeWarning: Mean of empty slice
  


  SUCCESS: 2017_Q1/VV_Q_mean.tif (min=0.01, max=370.63, mean=0.19)

Processing 2017 Q2 (3 images)...
  SUCCESS: 2017_Q2/VV_Q_mean.tif (min=0.01, max=357.03, mean=0.17)

Processing 2017 Q3 (3 images)...
  SUCCESS: 2017_Q3/VV_Q_mean.tif (min=0.01, max=394.69, mean=0.18)

Processing 2017 Q4 (3 images)...
  SUCCESS: 2017_Q4/VV_Q_mean.tif (min=0.01, max=435.46, mean=0.20)

Processing 2018 Q1 (3 images)...
  SUCCESS: 2018_Q1/VV_Q_mean.tif (min=0.01, max=414.39, mean=0.19)

Processing 2018 Q2 (3 images)...
  SUCCESS: 2018_Q2/VV_Q_mean.tif (min=0.01, max=386.81, mean=0.18)

Processing 2018 Q3 (3 images)...
  SUCCESS: 2018_Q3/VV_Q_mean.tif (min=0.01, max=391.82, mean=0.17)

Processing 2018 Q4 (3 images)...
  SUCCESS: 2018_Q4/VV_Q_mean.tif (min=0.01, max=402.45, mean=0.18)

Processing 2019 Q1 (3 images)...
  SUCCESS: 2019_Q1/VV_Q_mean.tif (min=0.01, max=419.64, mean=0.20)

Processing 2019 Q2 (3 images)...
  SUCCESS: 2019_Q2/VV_Q_mean.tif (min=0.01, max=418.67, mean=0.18)

Processing 2019 Q3 (3 i

In [7]:
vv_quarterly_median = calculate_quarterly_means_tree(
    in_root=s1_mosaic_10m_res__dir,
    out_root=s1_Q_mosaic__dir,
    band_pattern="VV*.tif",
    output_prefix="VV_Q",
    date_format="%Y_%m",
    aggregation="median",
)


Found 36 quarters with data:
  2017 Q1: 3 images
  2017 Q2: 3 images
  2017 Q3: 3 images
  2017 Q4: 3 images
  2018 Q1: 3 images
  2018 Q2: 3 images
  2018 Q3: 3 images
  2018 Q4: 3 images
  2019 Q1: 3 images
  2019 Q2: 3 images
  2019 Q3: 3 images
  2019 Q4: 3 images
  2020 Q1: 3 images
  2020 Q2: 3 images
  2020 Q3: 3 images
  2020 Q4: 3 images
  2021 Q1: 3 images
  2021 Q2: 3 images
  2021 Q3: 3 images
  2021 Q4: 3 images
  2022 Q1: 3 images
  2022 Q2: 3 images
  2022 Q3: 3 images
  2022 Q4: 3 images
  2023 Q1: 3 images
  2023 Q2: 3 images
  2023 Q3: 3 images
  2023 Q4: 3 images
  2024 Q1: 3 images
  2024 Q2: 3 images
  2024 Q3: 3 images
  2024 Q4: 3 images
  2025 Q1: 3 images
  2025 Q2: 3 images
  2025 Q3: 3 images
  2025 Q4: 2 images

Processing 2017 Q1 (3 images)...


C:\Users\NL1G3K\Desktop\Vegetation_quality_monitoring\notebooks_dir\_01_pre_processing_sat_obs\_support\_n04_funcs.py:875: RuntimeWarning: All-NaN slice encountered
  # Extract stem and add suffix


  SUCCESS: 2017_Q1/VV_Q_median.tif (min=0.01, max=406.24, mean=0.19)

Processing 2017 Q2 (3 images)...
  SUCCESS: 2017_Q2/VV_Q_median.tif (min=0.01, max=343.86, mean=0.17)

Processing 2017 Q3 (3 images)...
  SUCCESS: 2017_Q3/VV_Q_median.tif (min=0.01, max=363.53, mean=0.18)

Processing 2017 Q4 (3 images)...
  SUCCESS: 2017_Q4/VV_Q_median.tif (min=0.01, max=503.04, mean=0.20)

Processing 2018 Q1 (3 images)...
  SUCCESS: 2018_Q1/VV_Q_median.tif (min=0.01, max=397.55, mean=0.18)

Processing 2018 Q2 (3 images)...
  SUCCESS: 2018_Q2/VV_Q_median.tif (min=0.01, max=393.82, mean=0.18)

Processing 2018 Q3 (3 images)...
  SUCCESS: 2018_Q3/VV_Q_median.tif (min=0.01, max=390.05, mean=0.17)

Processing 2018 Q4 (3 images)...
  SUCCESS: 2018_Q4/VV_Q_median.tif (min=0.01, max=383.67, mean=0.18)

Processing 2019 Q1 (3 images)...
  SUCCESS: 2019_Q1/VV_Q_median.tif (min=0.01, max=424.31, mean=0.19)

Processing 2019 Q2 (3 images)...
  SUCCESS: 2019_Q2/VV_Q_median.tif (min=0.01, max=429.32, mean=0.18)

Pro

In [8]:
# Convert quarterly VV means to dB
processed, skipped = convert_quarterly_to_db(
    in_root=s1_Q_mosaic__dir,
    out_root=s1_Q_mosaic_dB__dir,
    pattern="VV_Q_mean.tif",
    pattern_out="*_dB.tif",
)


Found 36 files matching 'VV_Q_mean.tif'
  SUCCESS: VV_Q_mean.tif -> VV_Q_mean_dB.tif (min=-20.31dB, max=25.69dB, mean=-8.78dB)
  SUCCESS: VV_Q_mean.tif -> VV_Q_mean_dB.tif (min=-21.64dB, max=25.53dB, mean=-9.38dB)
  SUCCESS: VV_Q_mean.tif -> VV_Q_mean_dB.tif (min=-21.20dB, max=25.96dB, mean=-9.03dB)
  SUCCESS: VV_Q_mean.tif -> VV_Q_mean_dB.tif (min=-20.82dB, max=26.39dB, mean=-8.41dB)
  SUCCESS: VV_Q_mean.tif -> VV_Q_mean_dB.tif (min=-20.50dB, max=26.17dB, mean=-8.74dB)
  SUCCESS: VV_Q_mean.tif -> VV_Q_mean_dB.tif (min=-21.02dB, max=25.87dB, mean=-9.09dB)
  SUCCESS: VV_Q_mean.tif -> VV_Q_mean_dB.tif (min=-21.93dB, max=25.93dB, mean=-9.31dB)
  SUCCESS: VV_Q_mean.tif -> VV_Q_mean_dB.tif (min=-21.30dB, max=26.05dB, mean=-9.02dB)
  SUCCESS: VV_Q_mean.tif -> VV_Q_mean_dB.tif (min=-21.12dB, max=26.23dB, mean=-8.67dB)
  SUCCESS: VV_Q_mean.tif -> VV_Q_mean_dB.tif (min=-21.80dB, max=26.22dB, mean=-9.21dB)
  SUCCESS: VV_Q_mean.tif -> VV_Q_mean_dB.tif (min=-22.04dB, max=25.83dB, mean=-9.09dB)
  

## 2) VH_mean per Q

In [9]:
vh_quarterly_mean = calculate_quarterly_means_tree(
    in_root=s1_mosaic_10m_res__dir,
    out_root=s1_Q_mosaic__dir,
    band_pattern="VH*.tif",
    output_prefix="VH_Q",
    date_format="%Y_%m",
    aggregation="mean",
)


Found 36 quarters with data:
  2017 Q1: 3 images
  2017 Q2: 3 images
  2017 Q3: 3 images
  2017 Q4: 3 images
  2018 Q1: 3 images
  2018 Q2: 3 images
  2018 Q3: 3 images
  2018 Q4: 3 images
  2019 Q1: 3 images
  2019 Q2: 3 images
  2019 Q3: 3 images
  2019 Q4: 3 images
  2020 Q1: 3 images
  2020 Q2: 3 images
  2020 Q3: 3 images
  2020 Q4: 3 images
  2021 Q1: 3 images
  2021 Q2: 3 images
  2021 Q3: 3 images
  2021 Q4: 3 images
  2022 Q1: 3 images
  2022 Q2: 3 images
  2022 Q3: 3 images
  2022 Q4: 3 images
  2023 Q1: 3 images
  2023 Q2: 3 images
  2023 Q3: 3 images
  2023 Q4: 3 images
  2024 Q1: 3 images
  2024 Q2: 3 images
  2024 Q3: 3 images
  2024 Q4: 3 images
  2025 Q1: 3 images
  2025 Q2: 3 images
  2025 Q3: 3 images
  2025 Q4: 2 images

Processing 2017 Q1 (3 images)...
  SUCCESS: 2017_Q1/VH_Q_mean.tif (min=0.00, max=61.33, mean=0.04)

Processing 2017 Q2 (3 images)...
  SUCCESS: 2017_Q2/VH_Q_mean.tif (min=0.00, max=55.28, mean=0.04)

Processing 2017 Q3 (3 images)...
  SUCCESS: 2017_

In [10]:
vh_quarterly_median = calculate_quarterly_means_tree(
    in_root=s1_mosaic_10m_res__dir,
    out_root=s1_Q_mosaic__dir,
    band_pattern="VH*.tif",
    output_prefix="VH_Q",
    date_format="%Y_%m",
    aggregation="median",
)


Found 36 quarters with data:
  2017 Q1: 3 images
  2017 Q2: 3 images
  2017 Q3: 3 images
  2017 Q4: 3 images
  2018 Q1: 3 images
  2018 Q2: 3 images
  2018 Q3: 3 images
  2018 Q4: 3 images
  2019 Q1: 3 images
  2019 Q2: 3 images
  2019 Q3: 3 images
  2019 Q4: 3 images
  2020 Q1: 3 images
  2020 Q2: 3 images
  2020 Q3: 3 images
  2020 Q4: 3 images
  2021 Q1: 3 images
  2021 Q2: 3 images
  2021 Q3: 3 images
  2021 Q4: 3 images
  2022 Q1: 3 images
  2022 Q2: 3 images
  2022 Q3: 3 images
  2022 Q4: 3 images
  2023 Q1: 3 images
  2023 Q2: 3 images
  2023 Q3: 3 images
  2023 Q4: 3 images
  2024 Q1: 3 images
  2024 Q2: 3 images
  2024 Q3: 3 images
  2024 Q4: 3 images
  2025 Q1: 3 images
  2025 Q2: 3 images
  2025 Q3: 3 images
  2025 Q4: 2 images

Processing 2017 Q1 (3 images)...
  SUCCESS: 2017_Q1/VH_Q_median.tif (min=0.00, max=63.63, mean=0.04)

Processing 2017 Q2 (3 images)...
  SUCCESS: 2017_Q2/VH_Q_median.tif (min=0.00, max=55.74, mean=0.04)

Processing 2017 Q3 (3 images)...
  SUCCESS: 2

In [11]:
# Convert quarterly VH means to dB
processed, skipped = convert_quarterly_to_db(
    in_root=s1_Q_mosaic__dir,
    out_root=s1_Q_mosaic_dB__dir,
    pattern="VH_Q_mean.tif",
    pattern_out="*_dB.tif",
)


Found 36 files matching 'VH_Q_mean.tif'
  SUCCESS: VH_Q_mean.tif -> VH_Q_mean_dB.tif (min=-27.32dB, max=17.88dB, mean=-15.23dB)
  SUCCESS: VH_Q_mean.tif -> VH_Q_mean_dB.tif (min=-27.53dB, max=17.43dB, mean=-15.41dB)
  SUCCESS: VH_Q_mean.tif -> VH_Q_mean_dB.tif (min=-27.40dB, max=17.93dB, mean=-15.04dB)
  SUCCESS: VH_Q_mean.tif -> VH_Q_mean_dB.tif (min=-27.03dB, max=18.20dB, mean=-14.65dB)
  SUCCESS: VH_Q_mean.tif -> VH_Q_mean_dB.tif (min=-27.67dB, max=17.88dB, mean=-15.30dB)
  SUCCESS: VH_Q_mean.tif -> VH_Q_mean_dB.tif (min=-30.38dB, max=17.72dB, mean=-15.39dB)
  SUCCESS: VH_Q_mean.tif -> VH_Q_mean_dB.tif (min=-31.81dB, max=17.64dB, mean=-15.70dB)
  SUCCESS: VH_Q_mean.tif -> VH_Q_mean_dB.tif (min=-31.28dB, max=17.82dB, mean=-15.58dB)
  SUCCESS: VH_Q_mean.tif -> VH_Q_mean_dB.tif (min=-30.88dB, max=17.70dB, mean=-15.31dB)
  SUCCESS: VH_Q_mean.tif -> VH_Q_mean_dB.tif (min=-32.07dB, max=18.07dB, mean=-15.70dB)
  SUCCESS: VH_Q_mean.tif -> VH_Q_mean_dB.tif (min=-31.73dB, max=17.29dB, mean=-

## 3) Calculate the quarterly VV/VH ratio

### -> Option 1: VV/VH ratio of VV_mean/VH_mean

In [9]:
processed, skipped = calculate_vv_vh_ratio_tree(
    in_root=s1_Q_mosaic__dir,
    out_root=s1_Q_mosaic__dir,
    vv_pattern="VV_Q_mean.tif",
    vh_pattern="VH_Q_mean.tif",
    output_suffix="_ratio_Q_opt1",
)


SUCCESS 2017_Q1: VV_Q_mean.tif / VH_Q_mean.tif -> VV_VH_ratio_Q_opt1.tif (min=0.10, max=2865.41, mean=5.13)

SUCCESS 2017_Q2: VV_Q_mean.tif / VH_Q_mean.tif -> VV_VH_ratio_Q_opt1.tif (min=0.09, max=3109.08, mean=4.53)

SUCCESS 2017_Q3: VV_Q_mean.tif / VH_Q_mean.tif -> VV_VH_ratio_Q_opt1.tif (min=0.09, max=3271.81, mean=4.48)

SUCCESS 2017_Q4: VV_Q_mean.tif / VH_Q_mean.tif -> VV_VH_ratio_Q_opt1.tif (min=0.09, max=2841.49, mean=4.88)

SUCCESS 2018_Q1: VV_Q_mean.tif / VH_Q_mean.tif -> VV_VH_ratio_Q_opt1.tif (min=0.09, max=4662.04, mean=5.30)

SUCCESS 2018_Q2: VV_Q_mean.tif / VH_Q_mean.tif -> VV_VH_ratio_Q_opt1.tif (min=0.09, max=3697.59, mean=4.87)

SUCCESS 2018_Q3: VV_Q_mean.tif / VH_Q_mean.tif -> VV_VH_ratio_Q_opt1.tif (min=0.08, max=3583.05, mean=4.92)

SUCCESS 2018_Q4: VV_Q_mean.tif / VH_Q_mean.tif -> VV_VH_ratio_Q_opt1.tif (min=0.09, max=3740.18, mean=5.30)

SUCCESS 2019_Q1: VV_Q_mean.tif / VH_Q_mean.tif -> VV_VH_ratio_Q_opt1.tif (min=0.09, max=3011.40, mean=5.47)

SUCCESS 2019_Q2: V

In [13]:
processed, skipped = convert_quarterly_to_db(
    in_root=s1_Q_mosaic__dir,
    out_root=s1_Q_mosaic_dB__dir,
    pattern="*_ratio_Q_opt1.tif",
    pattern_out="*_dB.tif",
)


Found 36 files matching '*_ratio_Q_opt1.tif'
  SUCCESS: VV_VH_ratio_Q_opt1.tif -> VV_VH_ratio_Q_opt1_dB.tif (min=-9.95dB, max=34.57dB, mean=6.44dB)
  SUCCESS: VV_VH_ratio_Q_opt1.tif -> VV_VH_ratio_Q_opt1_dB.tif (min=-10.32dB, max=34.93dB, mean=6.03dB)
  SUCCESS: VV_VH_ratio_Q_opt1.tif -> VV_VH_ratio_Q_opt1_dB.tif (min=-10.45dB, max=35.15dB, mean=6.02dB)
  SUCCESS: VV_VH_ratio_Q_opt1.tif -> VV_VH_ratio_Q_opt1_dB.tif (min=-10.33dB, max=34.54dB, mean=6.24dB)
  SUCCESS: VV_VH_ratio_Q_opt1.tif -> VV_VH_ratio_Q_opt1_dB.tif (min=-10.62dB, max=36.69dB, mean=6.57dB)
  SUCCESS: VV_VH_ratio_Q_opt1.tif -> VV_VH_ratio_Q_opt1_dB.tif (min=-10.59dB, max=35.68dB, mean=6.30dB)
  SUCCESS: VV_VH_ratio_Q_opt1.tif -> VV_VH_ratio_Q_opt1_dB.tif (min=-10.91dB, max=35.54dB, mean=6.39dB)
  SUCCESS: VV_VH_ratio_Q_opt1.tif -> VV_VH_ratio_Q_opt1_dB.tif (min=-10.46dB, max=35.73dB, mean=6.56dB)
  SUCCESS: VV_VH_ratio_Q_opt1.tif -> VV_VH_ratio_Q_opt1_dB.tif (min=-10.51dB, max=34.79dB, mean=6.64dB)
  SUCCESS: VV_VH_ra

### -> Option 2: VV/VH ratio of each month and then avg that ratio to Q

In [14]:
processed, skipped = calculate_vv_vh_ratio_tree(
    in_root=s1_mosaic_10m_res__dir,
    out_root=s1_mosaic_10m_res__dir,
    vv_pattern="VV_10m.tif",
    vh_pattern="VH_10m.tif",
    output_suffix="_ratio",
)


SUCCESS 2017_01: VV_10m.tif / VH_10m.tif -> VV_VH_ratio.tif (min=0.13, max=3872.67, mean=5.21)

SUCCESS 2017_02: VV_10m.tif / VH_10m.tif -> VV_VH_ratio.tif (min=0.09, max=2570.89, mean=5.23)

SUCCESS 2017_03: VV_10m.tif / VH_10m.tif -> VV_VH_ratio.tif (min=0.09, max=2502.75, mean=5.03)

SUCCESS 2017_04: VV_10m.tif / VH_10m.tif -> VV_VH_ratio.tif (min=0.08, max=3671.16, mean=4.68)

SUCCESS 2017_05: VV_10m.tif / VH_10m.tif -> VV_VH_ratio.tif (min=0.10, max=3095.53, mean=4.60)

SUCCESS 2017_06: VV_10m.tif / VH_10m.tif -> VV_VH_ratio.tif (min=0.10, max=3373.50, mean=4.52)

SUCCESS 2017_07: VV_10m.tif / VH_10m.tif -> VV_VH_ratio.tif (min=0.10, max=4034.42, mean=4.42)

SUCCESS 2017_08: VV_10m.tif / VH_10m.tif -> VV_VH_ratio.tif (min=0.09, max=3318.06, mean=4.43)

SUCCESS 2017_09: VV_10m.tif / VH_10m.tif -> VV_VH_ratio.tif (min=0.08, max=2983.30, mean=4.64)

SUCCESS 2017_10: VV_10m.tif / VH_10m.tif -> VV_VH_ratio.tif (min=0.09, max=3314.34, mean=4.79)

SUCCESS 2017_11: VV_10m.tif / VH_10m.ti

In [15]:
vh_quarterly_mean = calculate_quarterly_means_tree(
    in_root=s1_mosaic_10m_res__dir,
    out_root=s1_Q_mosaic__dir,
    band_pattern="VV_VH_ratio.tif",
    output_prefix="VV_VH_ratio_Q_opt2",
    date_format="%Y_%m",
    aggregation="mean",
)


Found 36 quarters with data:
  2017 Q1: 3 images
  2017 Q2: 3 images
  2017 Q3: 3 images
  2017 Q4: 3 images
  2018 Q1: 3 images
  2018 Q2: 3 images
  2018 Q3: 3 images
  2018 Q4: 3 images
  2019 Q1: 3 images
  2019 Q2: 3 images
  2019 Q3: 3 images
  2019 Q4: 3 images
  2020 Q1: 3 images
  2020 Q2: 3 images
  2020 Q3: 3 images
  2020 Q4: 3 images
  2021 Q1: 3 images
  2021 Q2: 3 images
  2021 Q3: 3 images
  2021 Q4: 3 images
  2022 Q1: 3 images
  2022 Q2: 3 images
  2022 Q3: 3 images
  2022 Q4: 3 images
  2023 Q1: 3 images
  2023 Q2: 3 images
  2023 Q3: 3 images
  2023 Q4: 3 images
  2024 Q1: 3 images
  2024 Q2: 3 images
  2024 Q3: 3 images
  2024 Q4: 3 images
  2025 Q1: 3 images
  2025 Q2: 3 images
  2025 Q3: 3 images
  2025 Q4: 2 images

Processing 2017 Q1 (3 images)...


C:\Users\NL1G3K\Desktop\Vegetation_quality_monitoring\notebooks_dir\_01_pre_processing_sat_obs\_support\_n04_funcs.py:1103: RuntimeWarning: Mean of empty slice
  result = np.nanmean(stacked, axis=0)


  SUCCESS: 2017_Q1/VV_VH_ratio_Q_opt2_mean.tif (min=0.10, max=2935.01, mean=5.16)

Processing 2017 Q2 (3 images)...
  SUCCESS: 2017_Q2/VV_VH_ratio_Q_opt2_mean.tif (min=0.09, max=3107.94, mean=4.60)

Processing 2017 Q3 (3 images)...
  SUCCESS: 2017_Q3/VV_VH_ratio_Q_opt2_mean.tif (min=0.09, max=3284.40, mean=4.50)

Processing 2017 Q4 (3 images)...
  SUCCESS: 2017_Q4/VV_VH_ratio_Q_opt2_mean.tif (min=0.09, max=2878.21, mean=4.90)

Processing 2018 Q1 (3 images)...
  SUCCESS: 2018_Q1/VV_VH_ratio_Q_opt2_mean.tif (min=0.09, max=4439.21, mean=5.37)

Processing 2018 Q2 (3 images)...
  SUCCESS: 2018_Q2/VV_VH_ratio_Q_opt2_mean.tif (min=0.09, max=3712.59, mean=4.93)

Processing 2018 Q3 (3 images)...
  SUCCESS: 2018_Q3/VV_VH_ratio_Q_opt2_mean.tif (min=0.08, max=3604.67, mean=4.96)

Processing 2018 Q4 (3 images)...
  SUCCESS: 2018_Q4/VV_VH_ratio_Q_opt2_mean.tif (min=0.09, max=3874.01, mean=5.35)

Processing 2019 Q1 (3 images)...
  SUCCESS: 2019_Q1/VV_VH_ratio_Q_opt2_mean.tif (min=0.09, max=3012.64, m

In [19]:
processed, skipped = convert_quarterly_to_db(
    in_root=s1_Q_mosaic__dir,
    out_root=s1_Q_mosaic_dB__dir,
    pattern="VV_VH_ratio_Q_opt2_mean.tif",
    pattern_out="VV_VH_ratio_Q_opt2_dB.tif",
)


Found 36 files matching 'VV_VH_ratio_Q_opt2_mean.tif'
  SUCCESS: VV_VH_ratio_Q_opt2_mean.tif -> VV_VH_ratio_Q_opt2_dB.tif (min=-9.83dB, max=34.68dB, mean=6.47dB)
  SUCCESS: VV_VH_ratio_Q_opt2_mean.tif -> VV_VH_ratio_Q_opt2_dB.tif (min=-10.31dB, max=34.92dB, mean=6.08dB)
  SUCCESS: VV_VH_ratio_Q_opt2_mean.tif -> VV_VH_ratio_Q_opt2_dB.tif (min=-10.44dB, max=35.16dB, mean=6.04dB)
  SUCCESS: VV_VH_ratio_Q_opt2_mean.tif -> VV_VH_ratio_Q_opt2_dB.tif (min=-10.27dB, max=34.59dB, mean=6.26dB)
  SUCCESS: VV_VH_ratio_Q_opt2_mean.tif -> VV_VH_ratio_Q_opt2_dB.tif (min=-10.62dB, max=36.47dB, mean=6.62dB)
  SUCCESS: VV_VH_ratio_Q_opt2_mean.tif -> VV_VH_ratio_Q_opt2_dB.tif (min=-10.58dB, max=35.70dB, mean=6.35dB)
  SUCCESS: VV_VH_ratio_Q_opt2_mean.tif -> VV_VH_ratio_Q_opt2_dB.tif (min=-10.90dB, max=35.57dB, mean=6.42dB)
  SUCCESS: VV_VH_ratio_Q_opt2_mean.tif -> VV_VH_ratio_Q_opt2_dB.tif (min=-10.45dB, max=35.88dB, mean=6.60dB)
  SUCCESS: VV_VH_ratio_Q_opt2_mean.tif -> VV_VH_ratio_Q_opt2_dB.tif (min=-

## 4) VV std

In [21]:
processed, skipped = convert_monthly_to_db(
    in_root=s1_mosaic_10m_res__dir,
    out_root=s1_mosaic_10m_res__dir,
    pattern="VV_10m.tif",
    pattern_out="VV_10m_dB.tif",
)


Found 107 files matching 'VV_10m.tif'
  SUCCESS: VV_10m.tif -> VV_10m_dB.tif (min=-21.36dB, max=26.18dB, mean=-9.26dB)
  SUCCESS: VV_10m.tif -> VV_10m_dB.tif (min=-21.05dB, max=26.17dB, mean=-8.66dB)
  SUCCESS: VV_10m.tif -> VV_10m_dB.tif (min=-21.31dB, max=26.27dB, mean=-8.51dB)
  SUCCESS: VV_10m.tif -> VV_10m_dB.tif (min=-21.78dB, max=25.98dB, mean=-9.48dB)
  SUCCESS: VV_10m.tif -> VV_10m_dB.tif (min=-21.68dB, max=25.36dB, mean=-9.50dB)
  SUCCESS: VV_10m.tif -> VV_10m_dB.tif (min=-21.63dB, max=25.89dB, mean=-9.25dB)
  SUCCESS: VV_10m.tif -> VV_10m_dB.tif (min=-21.39dB, max=25.61dB, mean=-8.99dB)
  SUCCESS: VV_10m.tif -> VV_10m_dB.tif (min=-22.28dB, max=25.56dB, mean=-9.17dB)
  SUCCESS: VV_10m.tif -> VV_10m_dB.tif (min=-21.80dB, max=26.63dB, mean=-8.99dB)
  SUCCESS: VV_10m.tif -> VV_10m_dB.tif (min=-21.37dB, max=27.02dB, mean=-8.68dB)
  SUCCESS: VV_10m.tif -> VV_10m_dB.tif (min=-21.98dB, max=27.22dB, mean=-8.55dB)
  SUCCESS: VV_10m.tif -> VV_10m_dB.tif (min=-21.01dB, max=25.86dB, mea

In [22]:
out_paths = calculate_quarterly_std_tree(
    in_root=s1_mosaic_10m_res__dir,
    out_root=s1_Q_mosaic_dB__dir,
    band_pattern="VV_10m_dB.tif",
    output_prefix="VV_Q_std_dB",
    date_format="%Y_%m",
)


Found 36 quarters with data:
  2017 Q1: 3 months
  2017 Q2: 3 months
  2017 Q3: 3 months
  2017 Q4: 3 months
  2018 Q1: 3 months
  2018 Q2: 3 months
  2018 Q3: 3 months
  2018 Q4: 3 months
  2019 Q1: 3 months
  2019 Q2: 3 months
  2019 Q3: 3 months
  2019 Q4: 3 months
  2020 Q1: 3 months
  2020 Q2: 3 months
  2020 Q3: 3 months
  2020 Q4: 3 months
  2021 Q1: 3 months
  2021 Q2: 3 months
  2021 Q3: 3 months
  2021 Q4: 3 months
  2022 Q1: 3 months
  2022 Q2: 3 months
  2022 Q3: 3 months
  2022 Q4: 3 months
  2023 Q1: 3 months
  2023 Q2: 3 months
  2023 Q3: 3 months
  2023 Q4: 3 months
  2024 Q1: 3 months
  2024 Q2: 3 months
  2024 Q3: 3 months
  2024 Q4: 3 months
  2025 Q1: 3 months
  2025 Q2: 3 months
  2025 Q3: 3 months
  2025 Q4: 2 months

Processing 2017 Q1 (3 months)...


c:\Users\NL1G3K\Desktop\Vegetation_quality_monitoring\.pixi\envs\default\Lib\site-packages\numpy\lib\nanfunctions.py:1879: RuntimeWarning: Degrees of freedom <= 0 for slice.
  var = nanvar(a, axis=axis, dtype=dtype, out=out, ddof=ddof,


  SUCCESS: 2017_Q1/VV_Q_std_dB_std.tif (n=3 months, min=0.0002, max=15.6834, mean=0.5277)

Processing 2017 Q2 (3 months)...
  SUCCESS: 2017_Q2/VV_Q_std_dB_std.tif (n=3 months, min=0.0001, max=14.4657, mean=0.4873)

Processing 2017 Q3 (3 months)...
  SUCCESS: 2017_Q3/VV_Q_std_dB_std.tif (n=3 months, min=0.0001, max=17.6254, mean=0.4138)

Processing 2017 Q4 (3 months)...
  SUCCESS: 2017_Q4/VV_Q_std_dB_std.tif (n=3 months, min=0.0001, max=18.8573, mean=0.5271)

Processing 2018 Q1 (3 months)...
  SUCCESS: 2018_Q1/VV_Q_std_dB_std.tif (n=3 months, min=0.0001, max=24.8805, mean=0.7512)

Processing 2018 Q2 (3 months)...
  SUCCESS: 2018_Q2/VV_Q_std_dB_std.tif (n=3 months, min=0.0001, max=15.3345, mean=0.5499)

Processing 2018 Q3 (3 months)...
  SUCCESS: 2018_Q3/VV_Q_std_dB_std.tif (n=3 months, min=0.0002, max=18.3206, mean=0.5644)

Processing 2018 Q4 (3 months)...
  SUCCESS: 2018_Q4/VV_Q_std_dB_std.tif (n=3 months, min=0.0002, max=19.4532, mean=0.6610)

Processing 2019 Q1 (3 months)...
  SUCCES

## 5) VH std

In [23]:
processed, skipped = convert_monthly_to_db(
    in_root=s1_mosaic_10m_res__dir,
    out_root=s1_mosaic_10m_res__dir,
    pattern="VH_10m.tif",
    pattern_out="VH_10m_dB.tif",
)


Found 107 files matching 'VH_10m.tif'
  SUCCESS: VH_10m.tif -> VH_10m_dB.tif (min=-28.52dB, max=18.04dB, mean=-15.79dB)
  SUCCESS: VH_10m.tif -> VH_10m_dB.tif (min=-28.23dB, max=18.07dB, mean=-15.13dB)
  SUCCESS: VH_10m.tif -> VH_10m_dB.tif (min=-27.84dB, max=18.12dB, mean=-14.85dB)
  SUCCESS: VH_10m.tif -> VH_10m_dB.tif (min=-28.56dB, max=17.60dB, mean=-15.55dB)
  SUCCESS: VH_10m.tif -> VH_10m_dB.tif (min=-28.83dB, max=17.46dB, mean=-15.51dB)
  SUCCESS: VH_10m.tif -> VH_10m_dB.tif (min=-28.23dB, max=17.66dB, mean=-15.29dB)
  SUCCESS: VH_10m.tif -> VH_10m_dB.tif (min=-28.32dB, max=17.88dB, mean=-14.94dB)
  SUCCESS: VH_10m.tif -> VH_10m_dB.tif (min=-28.16dB, max=17.94dB, mean=-15.13dB)
  SUCCESS: VH_10m.tif -> VH_10m_dB.tif (min=-28.59dB, max=18.38dB, mean=-15.12dB)
  SUCCESS: VH_10m.tif -> VH_10m_dB.tif (min=-28.04dB, max=18.28dB, mean=-14.88dB)
  SUCCESS: VH_10m.tif -> VH_10m_dB.tif (min=-27.94dB, max=18.36dB, mean=-14.79dB)
  SUCCESS: VH_10m.tif -> VH_10m_dB.tif (min=-27.48dB, max=1

In [24]:
out_paths = calculate_quarterly_std_tree(
    in_root=s1_mosaic_10m_res__dir,
    out_root=s1_Q_mosaic_dB__dir,
    band_pattern="VH_10m_dB.tif",
    output_prefix="VH_Q_std_dB",
    date_format="%Y_%m",
)


Found 36 quarters with data:
  2017 Q1: 3 months
  2017 Q2: 3 months
  2017 Q3: 3 months
  2017 Q4: 3 months
  2018 Q1: 3 months
  2018 Q2: 3 months
  2018 Q3: 3 months
  2018 Q4: 3 months
  2019 Q1: 3 months
  2019 Q2: 3 months
  2019 Q3: 3 months
  2019 Q4: 3 months
  2020 Q1: 3 months
  2020 Q2: 3 months
  2020 Q3: 3 months
  2020 Q4: 3 months
  2021 Q1: 3 months
  2021 Q2: 3 months
  2021 Q3: 3 months
  2021 Q4: 3 months
  2022 Q1: 3 months
  2022 Q2: 3 months
  2022 Q3: 3 months
  2022 Q4: 3 months
  2023 Q1: 3 months
  2023 Q2: 3 months
  2023 Q3: 3 months
  2023 Q4: 3 months
  2024 Q1: 3 months
  2024 Q2: 3 months
  2024 Q3: 3 months
  2024 Q4: 3 months
  2025 Q1: 3 months
  2025 Q2: 3 months
  2025 Q3: 3 months
  2025 Q4: 2 months

Processing 2017 Q1 (3 months)...
  SUCCESS: 2017_Q1/VH_Q_std_dB_std.tif (n=3 months, min=0.0001, max=10.0628, mean=0.5786)

Processing 2017 Q2 (3 months)...
  SUCCESS: 2017_Q2/VH_Q_std_dB_std.tif (n=3 months, min=0.0001, max=11.8945, mean=0.5569)

P

# Done

End of the Notebook